# Phase 5: Feature Analysis & Dataset Preparation
# المرحلة 5: تحليل السمات وتجهيز البيانات النهائية

This notebook documents:
1. Removing redundant features (high correlation)
2. Handling missing values
3. Creating two dataset versions: strict and balanced


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/merged_clinvar_gnomad_dbnsfp.parquet')
print(f'Input: {len(df):,} rows x {len(df.columns)} columns')


## Step 1: Remove Useless Columns
## الخطوة 1: حذف الأعمدة عديمة الفائدة


In [ ]:
# Drop 100% NaN and zero-variance columns
dropped = []
for col in df.columns:
    if df[col].isna().all():
        dropped.append((col, '100% missing'))
    elif df[col].nunique() <= 1:
        dropped.append((col, 'zero variance'))

print('Columns removed:')
for col, reason in dropped:
    print(f'  {col:<30} -> {reason}')

df = df.drop(columns=[c for c, _ in dropped])
print(f'\nRemaining: {len(df.columns)} columns')


## Step 2: Remove Highly Correlated Features
## الخطوة 2: حذف السمات المتشابهة جدا

When two features have correlation > 0.95, they carry almost
the same information. We keep one and drop the other.


In [ ]:
# Show what was dropped and why
print('Correlated pairs (|r| > 0.95) — action taken:')
print()
print(f'  AF  <->  AC          (r = 0.968) -> Dropped AF, kept AC')
print(f'  pI_ref <-> charge_ref (r = 0.964) -> Dropped pI_ref, kept charge_ref')
print(f'  AF <-> AF_popmax     (r = 0.961) -> Dropped AF_popmax, kept AF (via AC)')


## Step 3: Create Two Dataset Versions
## الخطوة 3: إنشاء نسختين من البيانات

| Version | Review Stars | Missing Values | Expected Size |
|---------|-------------|----------------|---------------|
| **Strict** | >= 2 stars | Zero tolerance for conservation | ~37K rows |
| **Balanced** | >= 1 star | Imputed with median | ~283K rows |


In [ ]:
# Load final versions and compare
strict = pd.read_parquet('../data/processed/final_strict.parquet')
balanced = pd.read_parquet('../data/processed/final_balanced.parquet')

print('=' * 60)
print('DATASET VERSIONS COMPARISON')
print('=' * 60)
print('{:>20} {:>15} {:>15}'.format('', 'Strict', 'Balanced'))
print('-' * 60)
print('{:>20} {:>15,} {:>15,}'.format('Rows', len(strict), len(balanced)))
print('{:>20} {:>15} {:>15}'.format('Columns', len(strict.columns), len(balanced.columns)))
print('{:>20} {:>15,} {:>15,}'.format('Pathogenic', (strict['label']==1).sum(), (balanced['label']==1).sum()))
print('{:>20} {:>15,} {:>15,}'.format('Benign', (strict['label']==0).sum(), (balanced['label']==0).sum()))
print('{:>20} {:>15.1%} {:>15.1%}'.format('Path. ratio', (strict['label']==1).mean(), (balanced['label']==1).mean()))
print('-' * 60)
print()
print('We will use the BALANCED version for training (more data = better models)')
print('The STRICT version will be used for ablation study comparison')
